# Phase 1: Data Cleaning and Preprocessing

## StreamFlix Content Analytics Project

### Objective:
The objective of this phase is to inspect, clean, and preprocess the six StreamFlix datasets to ensure that the data is accurate, consistent, complete, and suitable for further analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
raw_data_path = "../data/raw"

subscribers_df = pd.read_csv(os.path.join(raw_data_path, "subscribers.csv"))
titles_df = pd.read_csv(os.path.join(raw_data_path, "titles.csv"))
watch_history_df = pd.read_csv(os.path.join(raw_data_path, "watch_history.csv"))
ratings_df = pd.read_csv(os.path.join(raw_data_path, "ratings.csv"))
reviews_df = pd.read_csv(os.path.join(raw_data_path, "reviews.csv"))
watchlist_df = pd.read_csv(os.path.join(raw_data_path, "watchlist.csv"))

print("All six datasets loaded successfully!")

All six datasets loaded successfully!


In [3]:
datasets = {
    "Subscribers": subscribers_df,
    "Titles": titles_df,
    "Watch History": watch_history_df,
    "Ratings": ratings_df,
    "Reviews": reviews_df,
    "Watchlist": watchlist_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Subscribers: 15,000 rows, 14 columns
Titles: 9,000 rows, 21 columns
Watch History: 650,000 rows, 10 columns
Ratings: 130,000 rows, 5 columns
Reviews: 110,000 rows, 7 columns
Watchlist: 65,000 rows, 5 columns


### Now let's calculate the total number of records across the entire StreamFlix dataset:

In [4]:
total_rows = sum(df.shape[0] for df in datasets.values())
total_columns = sum(df.shape[1] for df in datasets.values())

print(f"Total records across all datasets: {total_rows:,}")
print(f"Total columns across all datasets: {total_columns}")

Total records across all datasets: 979,000
Total columns across all datasets: 62


### Let's inspect column names and data types

In [5]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{name}")
    print(f"{'=' * 60}")
    print(df.dtypes.to_string())


Subscribers
subscriber_id            str
signup_date              str
country                  str
region                   str
age                    int64
gender                   str
plan_type                str
monthly_price_usd    float64
household_size         int64
primary_device           str
payment_method           str
tenure_months          int64
is_active               bool
churn_date               str

Titles
title_id                    str
title_name                  str
type                        str
primary_genre               str
country                     str
language                    str
release_year              int64
date_added                  str
maturity_rating             str
seasons                   int64
content_duration_min      int64
is_original                bool
license_type                str
director                    str
cast                        str
quality_score           float64
popularity_score        float64
license_cost_usd          int

#### Some columns may need conversion to datetime format. However, before changing anything, we'll inspect their actual values to check for inconsistent or invalid dates.

In [6]:
date_columns = {
    "Subscribers": (subscribers_df, ["signup_date", "churn_date"]),
    "Titles": (titles_df, ["date_added", "license_expiry"]),
    "Watch History": (watch_history_df, ["watch_date"]),
    "Ratings": (ratings_df, ["rating_date"]),
    "Reviews": (reviews_df, ["review_date"]),
    "Watchlist": (watchlist_df, ["added_date"])
}

for name, (df, columns) in date_columns.items():
    print(f"\n{name}")
    for column in columns:
        print(f"\n{column}:")
        print(df[column].dropna().head(6).to_list())


Subscribers

signup_date:
['2023-01-09', '2020-02-05', '2022-04-15', '2024-10-10', '2020-04-12', '2020-10-11']

churn_date:
['2022-12-08', '2021-12-15', '2024-10-19', '2020-07-22', '2026-05-08', '2025-08-06']

Titles

date_added:
['2026-04-05', '2026-05-01', '2026-04-17', '2018-12-08', '2016-05-11', '2023-09-23']

license_expiry:
['2028-08-03', '2027-02-16', '2029-10-11', '2028-11-21', '2029-04-30', '2027-11-08']

Watch History

watch_date:
['2024-01-01', '2025-10-18', '2026-04-26', '2026-03-09', '2023-12-27', '2026-05-06']

Ratings

rating_date:
['2026-02-06', '2025-03-21', '2024-05-26', '2026-05-22', '2026-05-16', '2026-05-26']

Reviews

review_date:
['2023-11-04', '2026-04-25', '2024-07-31', '2022-12-13', '2025-10-09', '2026-05-04']

Watchlist

added_date:
['2023-06-05', '2023-01-31', '2022-06-21', '2023-12-23', '2022-02-02', '2023-03-27']


#### The output shows that the date values across all six datasets follow a consistent YYYY-MM-DD format in the samples we inspected.

#### However, these are only sample values. We still need to check for invalid dates and missing values across the full datasets before converting the columns.

#### Now we'll check missing values in all six datasets

In [7]:
for name, df in datasets.items():
    print(f"\n{name}")
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print("No missing values found.")
    else:
        print(missing.to_string())


Subscribers
churn_date    11199

Titles
license_expiry    1966

Watch History
No missing values found.

Ratings
No missing values found.

Reviews
No missing values found.

Watchlist
No missing values found.


#### We should not fill or delete these missing values yet. In particular, an empty churn_date may be meaningful (Likely legitimate for subscribers who have not churned), while an empty license_expiry needs further investigation.

#### Now let's compare the missing churn_date values with is_active to see whether missing dates correspond to active subscribers.

In [8]:
pd.crosstab(
    subscribers_df["is_active"],
    subscribers_df["churn_date"].isna(),
    margins=True
)

churn_date,False,True,All
is_active,,,
False,3801,0,3801
True,0,11199,11199
All,3801,11199,15000


- All 11,199 active subscribers have a missing churn_date, which is expected because they haven't churned.
- All 3,801 inactive subscribers have a recorded churn_date.
- There are no mismatches between is_active and churn_date in these records.

Hence, we should preserve these missing values. They are meaningful and should not be filled with arbitrary dates or removed.

Now let's investigate the 1,966 missing values (license_expiry) in the titles dataset.

In [9]:
pd.crosstab(
    titles_df["is_original"],
    titles_df["license_expiry"].isna(),
    margins=True
)

license_expiry,False,True,All
is_original,,,
False,7034,0,7034
True,0,1966,1966
All,7034,1966,9000


#### The output confirms that all 1,966 missing license_expiry values belong to StreamFlix Original titles. This matches the project schema, which specifies that original titles can have a NULL licensing expiry date.

Conclusion:
- All 1,966 original titles have missing license_expiry values.
- All 7,034 non-original titles have recorded expiry dates.

#### Next, let's check whether any dataset contains completely duplicated rows.

In [10]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name}: {duplicate_count:,} duplicate rows")

Subscribers: 0 duplicate rows
Titles: 0 duplicate rows
Watch History: 0 duplicate rows
Ratings: 0 duplicate rows
Reviews: 0 duplicate rows
Watchlist: 0 duplicate rows


#### All six datasets have zero completely duplicated rows.

#### However, we still need to check duplicates in important ID columns separately. For example, two rows could have the same watch_id but differ in another column, so df.duplicated() wouldn't necessarily flag them. So, lets now check unique identifiers/IDs.

In [11]:
id_columns = {
    "Subscribers": (subscribers_df, "subscriber_id"),
    "Titles": (titles_df, "title_id"),
    "Watch History": (watch_history_df, "watch_id"),
    "Ratings": (ratings_df, "rating_id"),
    "Reviews": (reviews_df, "review_id"),
    "Watchlist": (watchlist_df, "watchlist_id")
}

for name, (df, column) in id_columns.items():
    duplicate_ids = df[column].duplicated().sum()
    print(f"{name}: {duplicate_ids:,} duplicate IDs")

Subscribers: 0 duplicate IDs
Titles: 0 duplicate IDs
Watch History: 0 duplicate IDs
Ratings: 0 duplicate IDs
Reviews: 0 duplicate IDs
Watchlist: 0 duplicate IDs


#### All six datasets have zero duplicate IDs, which means the identifier columns currently satisfy the uniqueness check.

#### Our earlier inspection showed that the date columns are stored as strings. The next step is to convert them into proper datetime values so we can analyze monthly trends, compare dates, and identify invalid date entries.

In [12]:
for name, (df, columns) in date_columns.items():
    for column in columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")
    print(f"{name}: date columns converted")

Subscribers: date columns converted
Titles: date columns converted
Watch History: date columns converted
Ratings: date columns converted
Reviews: date columns converted
Watchlist: date columns converted


All date columns across the six datasets have been processed into datetime format.

However, the message confirms that the conversion code ran; it doesn't yet confirm that every date was valid. We need to check whether any invalid date values were converted into NaT.

So let's, validate the converted date columns

In [13]:
for name, (df, columns) in date_columns.items():
    for column in columns:
        print(f"{name} - {column}: {df[column].isna().sum():,} missing dates")

Subscribers - signup_date: 0 missing dates
Subscribers - churn_date: 11,199 missing dates
Titles - date_added: 0 missing dates
Titles - license_expiry: 1,966 missing dates
Watch History - watch_date: 0 missing dates
Ratings - rating_date: 0 missing dates
Reviews - review_date: 0 missing dates
Watchlist - added_date: 0 missing dates


#### The results show that converting the date columns did not introduce any additional missing dates. The missing-value counts remain exactly as they were before conversion.

#### Now let's confirm that the conversion worked technically by checking the data types of the date columns.

In [14]:
for name, (df, columns) in date_columns.items():
    print(f"\n{name}")
    print(df[columns].dtypes.to_string())


Subscribers
signup_date    datetime64[us]
churn_date     datetime64[us]

Titles
date_added        datetime64[us]
license_expiry    datetime64[us]

Watch History
watch_date    datetime64[us]

Ratings
rating_date    datetime64[us]

Reviews
review_date    datetime64[us]

Watchlist
added_date    datetime64[us]


#### Next, we'll validate an important business rule from the project requirements: a churned subscriber's churn_date should be later than their signup_date.

In [15]:
churned_subscribers = subscribers_df[
    subscribers_df["is_active"] == False
]

invalid_churn_dates = churned_subscribers[
    churned_subscribers["churn_date"] <= churned_subscribers["signup_date"]
]

print(f"Churned subscribers: {len(churned_subscribers):,}")
print(f"Invalid churn dates: {len(invalid_churn_dates):,}")

Churned subscribers: 3,801
Invalid churn dates: 0


#### This means all 3,801 churned subscribers have a churn_date later than their signup_date. No date inconsistencies were found in this check.

#### Next, we'll check whether the watch_history dataset contains any records with a watch date earlier than the subscriber's signup date. Such records could indicate a data-quality issue.

In [16]:
watch_history_check = watch_history_df.merge(
    subscribers_df[["subscriber_id", "signup_date"]],
    on="subscriber_id",
    how="left"
)

invalid_watch_dates = watch_history_check[
    watch_history_check["watch_date"] < watch_history_check["signup_date"]
]

print(f"Total watch history records: {len(watch_history_check):,}")
print(f"Watch records before signup: {len(invalid_watch_dates):,}")

Total watch history records: 650,000
Watch records before signup: 0


#### This confirms that every watch history record has a watch_date on or after the corresponding subscriber's signup_date. No date inconsistencies were found.

#### Next, we'll check whether every subscriber_id and title_id in watch_history exists in the corresponding subscribers and titles datasets.

In [17]:
missing_subscribers = ~watch_history_df["subscriber_id"].isin(
    subscribers_df["subscriber_id"]
)

missing_titles = ~watch_history_df["title_id"].isin(
    titles_df["title_id"]
)

print(f"Watch records with invalid subscriber IDs: {missing_subscribers.sum():,}")
print(f"Watch records with invalid title IDs: {missing_titles.sum():,}")

Watch records with invalid subscriber IDs: 0
Watch records with invalid title IDs: 0


#### This confirms that every watch history record references a subscriber and a title that exist in their respective datasets.

#### Next, we'll check whether any records have invalid viewing durations, such as zero or negative values. This helps identify potentially incorrect watch-time data before calculating engagement KPIs.

In [18]:
invalid_durations = watch_history_df[
    watch_history_df["watch_duration_min"] <= 0
]

print(f"Total watch history records: {len(watch_history_df):,}")
print(f"Records with non-positive duration: {len(invalid_durations):,}")

Total watch history records: 650,000
Records with non-positive duration: 0


#### All 650,000 records have a positive watch duration. No records were flagged by this validation, so we don't need to correct anything for this check.

#### Next, we'll check whether completion_pct contains values outside the valid range of 0% to 100%.

In [19]:
invalid_completion = watch_history_df[
    (watch_history_df["completion_pct"] < 0) |
    (watch_history_df["completion_pct"] > 100)
]

print(f"Total watch history records: {len(watch_history_df):,}")
print(f"Records with invalid completion percentage: {len(invalid_completion):,}")

Total watch history records: 650,000
Records with invalid completion percentage: 0


#### All records have a completion_pct between 0% and 100%. No corrections are needed for this validation.

#### Now we'll check whether the completed column contains only valid Boolean values (True or False). This flag may be useful for analyzing completed titles and engagement patterns.

In [20]:
print(watch_history_df["completed"].value_counts(dropna=False))

completed
False    550637
True      99363
Name: count, dtype: int64


##### Both values are valid Boolean values, and all 650,000 records have a completion flag.

#### This confirms the values are valid, but we haven't yet checked whether the completed flag agrees with the completion_pct column.

#### So let's check consistency between completion status and percentage

In [21]:
inconsistent_completion = watch_history_df[
    (watch_history_df["completed"] & (watch_history_df["completion_pct"] < 100)) |
    (~watch_history_df["completed"] & (watch_history_df["completion_pct"] >= 100))
]

print(f"Records with inconsistent completion status: {len(inconsistent_completion):,}")

Records with inconsistent completion status: 98,712


##### 98,712 records have a potential inconsistency between completed and completion_pct. However, we should not modify anything yet. First, let's inspect the pattern to understand whether this is a genuine data-quality issue or simply how the dataset defines completion. So let's investigate the inconsistent records.

In [22]:
print(
    inconsistent_completion.groupby("completed")["completion_pct"]
    .agg(["count", "min", "max", "mean"])
)

           count   min   max       mean
completed                              
True       98712  90.0  99.9  94.845812


##### The flagged records all have completed = True, but their completion_pct values range from 90.0% to 99.9%, with an average of 94.85%.

##### This means our assumed rule that completed = True requires exactly 100% completion may not match how StreamFlix defines a completed viewing session. We shouldn't label these records as confirmed errors yet.

##### Let's check the completion threshold

In [23]:
print(watch_history_df.groupby("completed")["completion_pct"].describe())

              count       mean        std   min   25%   50%   75%    max
completed                                                               
False      550637.0  59.974977  20.384275   1.0  45.6  63.0  77.0   90.0
True        99363.0  94.879581   2.878831  90.0  92.4  94.8  97.3  100.0


##### Key finding: Every record marked completed = True has a completion percentage of at least 90%, while the maximum for records marked False is 90%.

##### This suggests that StreamFlix may use a completion threshold around 90%, rather than requiring exactly 100%. However, the data alone cannot confirm the intended business rule.

On page 3 of the Data Dictionary, the descriptions state:
- completion_pct: Percentage of the content completed, from 0 to 100.
- completed: Whether the viewing session passed the completion threshold.
On page 4 of the PRD, the required validation is different from the check we initially performed. It specifically asks us to verify whether:

completion_pct is approximately equal to watch_duration_min/content_duration_min x 100

The documentation does not specify that completed = True requires a completion percentage of exactly 100%. Therefore, our 98,712 flagged records are not necessarily data errors. So, let's validate completion percentage against viewing duration.

In [24]:
watch_history_check = watch_history_df[
    watch_history_df["content_duration_min"] > 0
].copy()

watch_history_check["calculated_completion_pct"] = (
    watch_history_check["watch_duration_min"] /
    watch_history_check["content_duration_min"]
) * 100

watch_history_check["completion_difference"] = (
    watch_history_check["completion_pct"] -
    watch_history_check["calculated_completion_pct"]
).abs()

print(watch_history_check["completion_difference"].describe())

count    650000.000000
mean          0.029353
std           0.021061
min           0.000000
25%           0.012500
50%           0.026531
75%           0.042373
max           0.155556
Name: completion_difference, dtype: float64


- The average difference is only about 0.029 percentage points.
- Even the largest observed difference is approximately 0.156 percentage points.
- These very small differences suggest that completion_pct closely matches the percentage calculated from watch_duration_min and content_duration_min.

This is likely due to rounding or precision differences. We have not identified a significant discrepancy in this check.

The PRD says the percentages should be approximately equal but doesn't define a tolerance. To quantify the result, let's use a provisional tolerance of 0.2 percentage points—slightly greater than the maximum difference we observed.

In [25]:
records_over_tolerance = watch_history_check[
    watch_history_check["completion_difference"] > 0.2
]

print(f"Records exceeding 0.2 percentage points: {len(records_over_tolerance):,}")

Records exceeding 0.2 percentage points: 0


- Records checked: 650,000
- Records exceeding the provisional tolerance of 0.2 percentage points: 0
- Maximum observed difference: approximately 0.156 percentage points

This confirms that all 650,000 records satisfy our chosen tolerance. The recorded completion percentages closely match the values calculated from viewing duration and content duration.

Next, Let's validate review sentiment values. The PRD specifically requires checking that reviews.sentiment contains only Positive, Neutral, or Negative.

In [26]:
print(reviews_df["sentiment"].value_counts(dropna=False))

sentiment
Positive    75007
Neutral     22759
Negative    12234
Name: count, dtype: int64


##### All 110,000 reviews belong to one of the three allowed sentiment categories specified in your Data Dictionary. No missing or unexpected sentiment values appear in the output.

##### Next, let's validate watchlist foreign key relationships. The PRD also requires checking whether every watchlist entry links to a real subscriber and a real title.

In [27]:
missing_watchlist_subscribers = ~watchlist_df["subscriber_id"].isin(
    subscribers_df["subscriber_id"]
)

missing_watchlist_titles = ~watchlist_df["title_id"].isin(
    titles_df["title_id"]
)

print(f"Watchlist entries with invalid subscriber IDs: {missing_watchlist_subscribers.sum():,}")
print(f"Watchlist entries with invalid title IDs: {missing_watchlist_titles.sum():,}")

Watchlist entries with invalid subscriber IDs: 0
Watchlist entries with invalid title IDs: 0


##### Now let's check whether watch duration exceeds content duration

In [28]:
duration_outliers = watch_history_df[
    watch_history_df["watch_duration_min"] >
    watch_history_df["content_duration_min"]
]

print(f"Total watch history records: {len(watch_history_df):,}")
print(f"Sessions exceeding content duration: {len(duration_outliers):,}")

Total watch history records: 650,000
Sessions exceeding content duration: 0


##### Next, we'll check whether the ratings dataset contains ratings within the valid 1–5 star range

In [29]:
invalid_ratings = ratings_df[
    (ratings_df["rating"] < 1) |
    (ratings_df["rating"] > 5)
]

print(f"Total rating records: {len(ratings_df):,}")
print(f"Ratings outside the 1–5 range: {len(invalid_ratings):,}")

Total rating records: 130,000
Ratings outside the 1–5 range: 0


##### Next, we'll check whether every review references an existing subscriber and an existing title. This helps ensure that reviews can be linked correctly during analysis.

In [30]:
missing_review_subscribers = ~reviews_df["subscriber_id"].isin(
    subscribers_df["subscriber_id"]
)

missing_review_titles = ~reviews_df["title_id"].isin(
    titles_df["title_id"]
)

print(f"Reviews with invalid subscriber IDs: {missing_review_subscribers.sum():,}")
print(f"Reviews with invalid title IDs: {missing_review_titles.sum():,}")

Reviews with invalid subscriber IDs: 0
Reviews with invalid title IDs: 0


##### All 110,000 reviews reference valid subscribers and titles. This confirms referential integrity for the reviews dataset.

##### Next, we'll perform the same check for the ratings dataset to ensure every rating is linked to an existing subscriber and title.

In [31]:
missing_rating_subscribers = ~ratings_df["subscriber_id"].isin(
    subscribers_df["subscriber_id"]
)

missing_rating_titles = ~ratings_df["title_id"].isin(
    titles_df["title_id"]
)

print(f"Ratings with invalid subscriber IDs: {missing_rating_subscribers.sum():,}")
print(f"Ratings with invalid title IDs: {missing_rating_titles.sum():,}")

Ratings with invalid subscriber IDs: 0
Ratings with invalid title IDs: 0


##### The ratings table has no invalid subscriber IDs or title IDs, meaning every rating is linked to an existing subscriber and title.

##### We already checked review relationships earlier, but let's now validate the watchlist table's primary key and duplicate records.

In [32]:
print("Duplicate watchlist IDs:", watchlist_df["watchlist_id"].duplicated().sum())
print("Duplicate watchlist rows:", watchlist_df.duplicated().sum())

Duplicate watchlist IDs: 0
Duplicate watchlist rows: 0


##### Before moving on, we should check the columns in watchlist_df to identify any additional validation rules required by the dataset.

In [33]:
print(watchlist_df.info())
print(watchlist_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 65000 entries, 0 to 64999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   watchlist_id   65000 non-null  int64         
 1   subscriber_id  65000 non-null  str           
 2   title_id       65000 non-null  str           
 3   added_date     65000 non-null  datetime64[us]
 4   watched        65000 non-null  bool          
dtypes: bool(1), datetime64[us](1), int64(1), str(2)
memory usage: 2.0 MB
None
   watchlist_id subscriber_id   title_id added_date  watched
0             1     SUB104004  TTL201827 2023-06-05     True
1             2     SUB102310  TTL201354 2023-01-31    False
2             3     SUB111860  TTL204871 2022-06-21     True
3             4     SUB113333  TTL205130 2023-12-23     True
4             5     SUB106850  TTL204396 2022-02-02    False


From the output, we can confirm:
- 65,000 records in the watchlist table.
- No missing values in any column.
- added_date has already been converted to datetime64[us].
- watched is correctly stored as a Boolean (True/False).
- Data types match the Data Dictionary.

Now let's validate is_active vs churn_date consistency (Final Business Rule Check)
One important business-rule validation remains for the Subscribers table.

A subscriber should satisfy exactly one of these conditions:
- Active subscriber → is_active = True and churn_date is missing.
- Churned subscriber → is_active = False and churn_date is present.

Although we already checked this using pd.crosstab(), we'll perform one final validation by directly identifying any inconsistent records.

In [34]:
invalid_subscriber_status = subscribers_df[
    (
        (subscribers_df["is_active"] == True) &
        (subscribers_df["churn_date"].notna())
    ) |
    (
        (subscribers_df["is_active"] == False) &
        (subscribers_df["churn_date"].isna())
    )
]

print(f"Subscribers with inconsistent active/churn status: {len(invalid_subscriber_status):,}")

Subscribers with inconsistent active/churn status: 0


##### Next, let's check whether any watchlist entry was added before the subscriber signed up. This is a useful chronological consistency check.

In [35]:
watchlist_check = watchlist_df.merge(
    subscribers_df[["subscriber_id", "signup_date"]],
    on="subscriber_id",
    how="left"
)

invalid_watchlist_dates = watchlist_check[
    watchlist_check["added_date"] < watchlist_check["signup_date"]
]

print(f"Watchlist entries before subscriber signup: {len(invalid_watchlist_dates):,}")

Watchlist entries before subscriber signup: 0


##### The result is 0 watchlist entries before subscriber signup. This means every watchlist entry was added on or after the corresponding subscriber's signup date. No corrections are needed.

##### We previously checked this and found 0 sessions where watch_duration_min exceeded content_duration_min. Since that check is already complete, let's move to another important validation: checking for negative or zero content durations.

In [36]:
invalid_content_durations = watch_history_df[
    watch_history_df["content_duration_min"] <= 0
]

print(f"Records with zero or negative content duration: {len(invalid_content_durations):,}")

Records with zero or negative content duration: 0


##### So, we've completed several validation checks. Before moving forward, let's confirm that our date conversions and validation work haven't introduced any unexpected missing values in the six datasets.

In [37]:
for name, df in datasets.items():
    print(f"{name}: {df.isna().sum().sum():,} missing values")

Subscribers: 11,199 missing values
Titles: 1,966 missing values
Watch History: 0 missing values
Ratings: 0 missing values
Reviews: 0 missing values
Watchlist: 0 missing values


##### The results match the missing-value checks we performed earlier.

##### Now let's create a summary of the six datasets for your Phase 1 notebook. This will help document the number of rows, columns, and missing values in each dataset.

In [38]:
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isna().sum().sum(),
        "Duplicate Rows": df.duplicated().sum()
    })

quality_summary_df = pd.DataFrame(quality_summary)

quality_summary_df

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Subscribers,15000,14,11199,0
1,Titles,9000,21,1966,0
2,Watch History,650000,10,0,0
3,Ratings,130000,5,0,0
4,Reviews,110000,7,0,0
5,Watchlist,65000,5,0,0


##### The table confirms that all six datasets have 979,000 records and 62 columns combined, with zero exact duplicate rows across all six datasets.

##### The 13,165 missing values are expected, based on our earlier checks of subscriber churn dates and title license expiry dates. We should preserve them rather than fill or delete them.

In [41]:
quality_summary_df.to_csv(
    "../outputs/reports/Phase1_Data_Quality_Summary.csv",
    index=False
)

Before moving to the next task, let's document the checks in your notebook.

The six StreamFlix datasets were inspected for missing values, duplicate records, data types, primary key uniqueness, and referential integrity.

- **Total records:** 979,000 across six datasets.
- **Total columns:** 62.
- **Duplicate rows:** No exact duplicate rows were found.
- **Missing values:** 13,165 missing values were identified, all in `subscribers.churn_date` and `titles.license_expiry`. These were validated as expected business exceptions and retained.
- **Referential integrity:** Foreign key checks performed on the relevant tables found no invalid references.
- **Business rules:** Subscriber churn dates, watch durations, completion percentages, review sentiment values, and watchlist dates were checked against the applicable rules.

The consolidated data quality summary was exported to `reports/Phase1_Data_Quality_Summary.csv` for use in the Phase 1 Data Quality Report.

##### So far, we've recorded the total missing values per dataset. Now we'll calculate the percentage of rows affected in each column that contains missing values.

In [42]:
for name, df in datasets.items():
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0]

    if not missing_counts.empty:
        missing_percentages = (missing_counts / len(df)) * 100

        print(f"\n{name}")
        print(pd.DataFrame({
            "Missing Values": missing_counts,
            "Percentage of Rows": missing_percentages.round(2)
        }))


Subscribers
            Missing Values  Percentage of Rows
churn_date           11199               74.66

Titles
                Missing Values  Percentage of Rows
license_expiry            1966               21.84


##### Next, the PRD specifically asks us to identify subscribers with very short tenure who are still active and note that these records are acceptable. It doesn't specify a threshold for "very short," so we'll first inspect the tenure distribution rather than arbitrarily label records as errors.

In [43]:
active_subscribers = subscribers_df[
    subscribers_df["is_active"] == True
].copy()

active_subscribers["tenure_days"] = (
    active_subscribers["signup_date"] - active_subscribers["signup_date"].min()
).dt.days

print(active_subscribers["tenure_days"].describe())

count    11199.000000
mean      2449.307795
std        941.235006
min          0.000000
25%       1766.000000
50%       2614.000000
75%       3251.000000
max       3780.000000
Name: tenure_days, dtype: float64


##### There's one important correction we need to make before interpreting these results.The output shows 11,199 active subscribers, and the date differences range from 0 to 3,780 days. However, the calculation measures days since the earliest signup date in the dataset, not each subscriber's actual tenure.

##### For active subscribers, we'll calculate tenure from their signup date to the latest watch_date in the dataset as a consistent reference point for this analysis.

In [44]:
reference_date = watch_history_df["watch_date"].max()

active_subscribers["tenure_days"] = (
    reference_date - active_subscribers["signup_date"]
).dt.days

print(active_subscribers["tenure_days"].describe())

count    11199.000000
mean      1330.692205
std        941.235006
min          0.000000
25%        529.000000
50%       1166.000000
75%       2014.000000
max       3780.000000
Name: tenure_days, dtype: float64


##### The median active subscriber has been registered for approximately 3.2 years as of the reference date. Some subscribers have a tenure of zero days, which could mean they signed up on the latest viewing date.

##### The PRD doesn't define a threshold for “very short tenure,” so we'll use 30 days as a provisional threshold for inspection (not as an official project rule yet.)

In [45]:
short_tenure_subscribers = active_subscribers[
    active_subscribers["tenure_days"] <= 30
]

print(f"Active subscribers with tenure of 30 days or less: {len(short_tenure_subscribers):,}")

Active subscribers with tenure of 30 days or less: 172


##### We found 172 active subscribers with a tenure of 30 days or less. That represents approximately 1.54% of the 11,199 active subscribers.

##### These subscribers have recently joined the platform and are still marked active. As specified in the project requirements, this is acceptable and should be documented rather than treated as a data quality error.

##### Using the latest viewing date as the reference date, 172 active subscribers had a tenure of 30 days or less. This represents approximately 1.54% of the 11,199 active subscribers. These records were retained because recently registered subscribers can legitimately remain active. The 30-day threshold was used for inspection because the project requirements do not specify a threshold for very short tenure.

##### Also, the PRD requires durations and prices to be numeric, so let's inspect the relevant columns in your six datasets before making any changes.

In [46]:
numeric_columns = {
    "Subscribers": (subscribers_df, ["age", "tenure_months", "monthly_price_usd"]),
    "Titles": (titles_df, ["release_year", "content_duration_min", "license_cost_usd"]),
    "Watch History": (watch_history_df, ["watch_duration_min", "content_duration_min", "completion_pct"]),
    "Ratings": (ratings_df, ["rating"]),
    "Reviews": (reviews_df, ["helpful_votes"]),
    "Watchlist": (watchlist_df, [])
}

for name, (df, columns) in numeric_columns.items():
    print(f"\n{name}")
    for column in columns:
        print(f"{column}: {df[column].dtype}")


Subscribers
age: int64
tenure_months: int64
monthly_price_usd: float64

Titles
release_year: int64
content_duration_min: int64
license_cost_usd: int64

Watch History
watch_duration_min: float64
content_duration_min: int64
completion_pct: float64

Ratings
rating: int64

Reviews
helpful_votes: int64

Watchlist


##### The numeric columns we inspected all have appropriate numeric data types.
- Subscribers: age and tenure are integers; monthly price is a decimal.
- Titles: release year and content duration are integers; license cost is numeric.
- Watch History: watch duration and completion percentage are decimals; content duration is an integer.
- Ratings: rating is an integer.
- Reviews: helpful votes are integers.
- Watchlist: no numeric columns were selected for this check.

No data type conversions are needed for these columns based on the output.

##### Let's check that the date columns are stored as dates and the status columns are stored as Boolean values.

In [47]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.dtypes[df.dtypes == "object"])


Subscribers
Series([], dtype: object)

Titles
Series([], dtype: object)

Watch History
Series([], dtype: object)

Ratings
Series([], dtype: object)

Reviews
Series([], dtype: object)

Watchlist
Series([], dtype: object)


##### All six datasets returned an empty object-type column list. This means none of the columns are currently stored as generic object types. That's a good sign, although this check alone doesn't prove that every date and Boolean column has the correct type.

##### Let's make one final targeted check of the expected date and Boolean columns

In [48]:
columns_to_check = {
    "Subscribers": (subscribers_df, ["signup_date", "churn_date", "is_active"]),
    "Titles": (titles_df, ["date_added", "license_expiry", "is_original"]),
    "Watch History": (watch_history_df, ["watch_date", "completed"]),
    "Ratings": (ratings_df, ["rating_date"]),
    "Reviews": (reviews_df, ["review_date"]),
    "Watchlist": (watchlist_df, ["added_date", "watched"])
}

for name, (df, columns) in columns_to_check.items():
    print(f"\n{name}")
    print(df[columns].dtypes)


Subscribers
signup_date    datetime64[us]
churn_date     datetime64[us]
is_active                bool
dtype: object

Titles
date_added        datetime64[us]
license_expiry    datetime64[us]
is_original                 bool
dtype: object

Watch History
watch_date    datetime64[us]
completed               bool
dtype: object

Ratings
rating_date    datetime64[us]
dtype: object

Reviews
review_date    datetime64[us]
dtype: object

Watchlist
added_date    datetime64[us]
watched                 bool
dtype: object


##### All expected date and Boolean columns have the correct data types. No additional conversions are needed for these columns.

##### Now we'll create the one-page Data Quality report required by the StreamFlix PRD. It will summarize the dataset, validation checks, findings, and treatment of expected missing values.

##### The Data Quality report is saved at Internmo_StreamFlix_Project\outputs\reports